# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models")
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [3]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


--2026-09-22 18:34:02--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.55, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-09-22 18:34:02 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]


KeyboardInterrupt: 

In [2]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-09-22 18:31:34--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 108.138.246.85, 108.138.246.71, 108.138.246.67, ...
Connecting to huggingface.co (huggingface.co)|108.138.246.85|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&Expires=1790105494&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD91c2VyX2lkPXB1YmxpYyZY

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [4]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

{'model_name': 'my_model',
 'target_phrase': ['hey jarvis'],
 'custom_negative_phrases': [],
 'n_samples': 10000,
 'n_samples_val': 2000,
 'tts_batch_size': 50,
 'augmentation_batch_size': 16,
 'piper_sample_generator_path': './piper-sample-generator',
 'output_dir': './my_custom_model',
 'rir_paths': ['./mit_rirs'],
 'background_paths': ['./background_clips'],
 'background_paths_duplication_rate': [1],
 'false_positive_validation_data_path': './validation_set_features.npy',
 'augmentation_rounds': 1,
 'feature_data_files': {'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
 'batch_n_per_class': {'ACAV100M_sample': 1024,
  'adversarial_negative': 50,
  'positive': 50},
 'model_type': 'dnn',
 'layer_size': 32,
 'steps': 50000,
 'max_negative_weight': 1500,
 'target_false_positives_per_hour': 0.2}

In [6]:
# Modify values in the config and save a new version

config["target_phrase"] = ["nova"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [8]:
!pip install openwakeword

  Using cached onnxruntime-1.30.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
INFO: pip is looking at multiple versions of openwakeword to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 80.1 MB/s eta 0:00:00


In [18]:
# 1. Uyumlu torchaudio sürümünü yükle
!pip install torchaudio==2.1.2 torch==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# 2. torch-audiomentations paketinin güncel halini kontrol et
!pip install --upgrade torch-audiomentations

!pip install generate_samples

Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torchaudio==2.1.2 (from versions: none)
ERROR: No matching distribution found for torchaudio==2.1.2
ERROR: Could not find a version that satisfies the requirement generate_samples (from versions: none)
ERROR: No matching distribution found for generate_samples


In [65]:
import os
import glob
import shutil
import yaml

# 1. Konfigürasyonu oku
config_path = "/content/my_model.yaml"
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

target_phrase = cfg.get("target_phrase", ["hey nova"])[0]
output_dir = cfg.get("output_dir", "/content")

dest_dirs = [
    os.path.join(output_dir, target_phrase),
    os.path.join("/content/openwakeword", target_phrase)
]

source_clips = glob.glob("/content/positive_clips/*.wav")
print(f"[+] Kaynakta bulunan klip sayısı: {len(source_clips)}")

# Sadece farklı klasörlere kopyala
for d in dest_dirs:
    os.makedirs(d, exist_ok=True)
    for f in source_clips:
        dst_path = os.path.join(d, os.path.basename(f))
        if os.path.abspath(f) != os.path.abspath(dst_path):
            shutil.copy(f, dst_path)

# 2. train.py dosyasında positive_clips listesini garantiye al
train_script = "/content/openwakeword/openwakeword/train.py"
with open(train_script, "r") as f:
    content = f.read()

target_str = "sr, dat = scipy.io.wavfile.read(positive_clips["
replacement = """if len(positive_clips) == 0:
        import glob
        positive_clips = glob.glob('/content/positive_clips/*.wav')
    sr, dat = scipy.io.wavfile.read(positive_clips["""

if target_str in content and "if len(positive_clips) == 0:" not in content:
    content = content.replace(target_str, replacement)
    with open(train_script, "w") as f:
        f.write(content)
    print("[+] train.py başarıyla yamalandı.")
else:
    print("[*] train.py zaten güncel.")

[+] Kaynakta bulunan klip sayısı: 50
[*] train.py zaten güncel.


In [69]:
import glob
import subprocess
import os

print("[+] MP3 formatındaki klipler 16kHz WAV formatına dönüştürülüyor...")

for wav_path in glob.glob("/content/positive_clips/*.wav"):
    temp_path = wav_path.replace(".wav", "_temp.wav")
    # 16kHz, mono, 16-bit PCM WAV olarak yeniden kodla
    cmd = f"ffmpeg -y -i {wav_path} -ar 16000 -ac 1 -c:a pcm_s16le {temp_path} -loglevel quiet"
    subprocess.run(cmd, shell=True, check=True)
    os.replace(temp_path, wav_path)

print("[+] Dönüştürme tamamlandı, tüm dosyalar gerçek RIFF WAV oldu!")

[+] MP3 formatındaki klipler 16kHz WAV formatına dönüştürülüyor...
[+] Dönüştürme tamamlandı, tüm dosyalar gerçek RIFF WAV oldu!


In [70]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!cd /content && PYTHONPATH=/content/openwakeword/openwakeword python /content/openwakeword/openwakeword/train.py --training_config /content/my_model.yaml --generate_clips

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
[+] '['hey nova']' icin 1000 adet sentetik ses uretimi tetiklendi...
[+] Hedef klasor: /content/my_custom_model/hey_nova/positive_train
[+] '['hey nova']' icin 1000 adet sentetik ses uretimi tetiklendi...
[+] Hedef klasor: /content/my_custom_model/hey_nova/positive_test
[+] '['stenerson stiefelhagen', 'hey novakowski', 'jahanian novick', 'gerstenhaber nova', 'lehane devotional', 'hey', 'haney', 'behaving privacies', 'hadler', 'nova hague', 'dillehay', 'espanola', 'approval', 'travelodge hazier', 'misbehaving stephens', 'haverstock koeppel', 'haydock doughman', 'revenuer hey', 'haywire hoboken', 'hey', 'novo', 'haid', 'ruminations hait', 'd

In [74]:
import os
import glob
import shutil

# Hedef klasör yolları
base_dir = "/content/my_custom_model/hey_nova"
pos_train = os.path.join(base_dir, "positive_train")
pos_test = os.path.join(base_dir, "positive_test")
neg_train = os.path.join(base_dir, "negative_train")
neg_test = os.path.join(base_dir, "negative_test")

for p in [pos_train, pos_test, neg_train, neg_test]:
    os.makedirs(p, exist_ok=True)

# 1. Pozitif klipleri doldur (/content/positive_clips içindeki dönüştürülmüş wav dosyaları)
clips = sorted(glob.glob("/content/positive_clips/*.wav"))
print(f"[+] Toplam pozitif klip: {len(clips)}")

# %80 train, %20 test olarak paylaştır
split_idx = int(len(clips) * 0.8)
train_clips = clips[:split_idx]
test_clips = clips[split_idx:]

for f in train_clips:
    shutil.copy(f, pos_train)
for f in test_clips:
    shutil.copy(f, pos_test)

# 2. Negatif klasörlerin de boş kalıp StopIteration vermemesi için audioset/sessizlikten dosya koy
sample_bg = "/content/audioset_16k/sample.wav"
if os.path.exists(sample_bg):
    shutil.copy(sample_bg, os.path.join(neg_train, "neg_1.wav"))
    shutil.copy(sample_bg, os.path.join(neg_test, "neg_1.wav"))

print(f"[+] positive_train dosya sayısı: {len(os.listdir(pos_train))}")
print(f"[+] positive_test dosya sayısı: {len(os.listdir(pos_test))}")
print(f"[+] negative_train dosya sayısı: {len(os.listdir(neg_train))}")
print(f"[+] negative_test dosya sayısı: {len(os.listdir(neg_test))}")

[+] Toplam pozitif klip: 50
[+] positive_train dosya sayısı: 40
[+] positive_test dosya sayısı: 10
[+] negative_train dosya sayısı: 1
[+] negative_test dosya sayısı: 1


In [78]:
# 1. train.py dosyasındaki argümanları incele
!grep -A 20 "ArgumentParser" /content/openwakeword/openwakeword/train.py

# 2. Dosyanın en altındaki ana çalıştırma mantığını (main / if __name__) kontrol et
!tail -n 35 /content/openwakeword/openwakeword/train.py

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--training_config",
        help="The path to the training config file (required)",
        type=str,
        required=True
    )
    parser.add_argument(
        "--generate_clips",
        help="Execute the synthetic data generation process",
        action="store_true",
        default="False",
        required=False
    )
    parser.add_argument(
        "--augment_clips",
        help="Execute the synthetic data augmentation process",
        action="store_true",
        default="False",
        required=False
    )
        X_val_fp_labels = np.zeros(X_val_fp.shape[0]).astype(np.float32)
        X_val_fp = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(torch.from_numpy(X_val_fp), torch.from_numpy(X_val_fp_labels)),
            batch_size=len(X_val_fp_labels)
        )

        X_val_pos = np.load(os.path.join(feature_save_dir, "positive_features_test.npy"))
        X_val_neg = np.load

In [88]:
train_script = "/content/openwakeword/openwakeword/train.py"
with open(train_script, "r", encoding="utf-8") as f:
    code = f.read()

# "features already exist" uyarısını veren bloğu baypas et
old_check = 'if os.path.exists(os.path.join(feature_save_dir, "positive_features_train.npy")):'
# Eğer dosya farklı bir isim kontrol ediyorsa genel satırı bul
if "Openwakeword features already exist" in code:
    lines = code.splitlines()
    new_lines = []
    skip = False
    for line in lines:
        if "Openwakeword features already exist" in line:
            # Bu uyarıyı veren bloğun başına pass ekle veya if'i False yap
            pass
        new_lines.append(line)
    print("[+] train.py kontrol edildi.")

# Alternatif ve en temiz yol: features kontrolünün sonucunu zorla False yapmak
code = code.replace("if os.path.exists(os.path.join(feature_save_dir,", "if False and os.path.exists(os.path.join(feature_save_dir,")
with open(train_script, "w", encoding="utf-8") as f:
    f.write(code)

print("[+] Atlatma (skipping) kontrolü devre dışı bırakıldı, özellik çıkarımı zorlanacak!")

[+] train.py kontrol edildi.
[+] Atlatma (skipping) kontrolü devre dışı bırakıldı, özellik çıkarımı zorlanacak!


In [99]:
# 1. train.py dosyasını orijinal haline sıfırla
# 1. train.py dosyasını orijinal haline sıfırla
!git -C /content/openwakeword checkout openwakeword/train.py

# 2. Önbellekte kalan TÜM .npy özellik dosyalarını kökten temizle
!rm -rf /content/my_custom_model/*.npy
!rm -rf /content/my_custom_model/*/*.npy
!rm -rf /content/*.npy

Updated 1 path from the index


In [100]:
# Step 2 Augment the clips

!cd /content && PYTHONPATH=/content/openwakeword python /content/openwakeword/openwakeword/train.py --training_config /content/my_model.yaml --augment_clips


/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_audiomentations/core/transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmente

In [83]:
!ls -lh /content/my_custom_model/hey_nova/

total 620K
-rw-r--r-- 1 root root 8.3K Sep 22 19:07 clip_0.wav
-rw-r--r-- 1 root root 9.4K Sep 22 19:07 clip_10.wav
-rw-r--r-- 1 root root 9.4K Sep 22 19:07 clip_11.wav
-rw-r--r-- 1 root root 9.4K Sep 22 19:07 clip_12.wav
-rw-r--r-- 1 root root 9.4K Sep 22 19:07 clip_13.wav
-rw-r--r-- 1 root root 9.4K Sep 22 19:07 clip_14.wav
-rw-r--r-- 1 root root  11K Sep 22 19:07 clip_15.wav
-rw-r--r-- 1 root root  11K Sep 22 19:07 clip_16.wav
-rw-r--r-- 1 root root  11K Sep 22 19:07 clip_17.wav
-rw-r--r-- 1 root root  11K Sep 22 19:07 clip_18.wav
-rw-r--r-- 1 root root  11K Sep 22 19:07 clip_19.wav
-rw-r--r-- 1 root root 8.3K Sep 22 19:07 clip_1.wav
-rw-r--r-- 1 root root 8.3K Sep 22 19:07 clip_20.wav
-rw-r--r-- 1 root root 8.3K Sep 22 19:07 clip_21.wav
-rw-r--r-- 1 root root 8.3K Sep 22 19:07 clip_22.wav
-rw-r--r-- 1 root root 8.3K Sep 22 19:07 clip_23.wav
-rw-r--r-- 1 root root 8.3K Sep 22 19:07 clip_24.wav
-rw-r--r-- 1 root root 9.6K Sep 22 19:07 clip_25.wav
-rw-r--r-- 1 root root 9.6K Sep 22 19

In [102]:
!wget -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy -O /content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

--2026-09-22 19:32:03--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.17, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&Expires=1790109123&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD9YLVhldC1DYXMtVWlkPXB1Ymxp

In [110]:
!grep -B 15 -A 5 "best_model = oww.auto_train" /content/openwakeword/openwakeword/train.py

        )

        X_val_pos = np.load(os.path.join(feature_save_dir, "positive_features_test.npy"))
        X_val_neg = np.load(os.path.join(feature_save_dir, "negative_features_test.npy"))
        labels = np.hstack((np.ones(X_val_pos.shape[0]), np.zeros(X_val_neg.shape[0]))).astype(np.float32)

        X_val = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(
                torch.from_numpy(np.vstack((X_val_pos, X_val_neg))),
                torch.from_numpy(labels)
                ),
            batch_size=len(labels)
        )

        # Run auto training
        best_model = oww.auto_train(
            X_train=X_train,
            X_val=X_val,
            false_positive_val_data=X_val_fp,
            steps=config["steps"],
            max_negative_weight=config["max_negative_weight"],


In [163]:
!sed -n '75,85p' /content/openwakeword/openwakeword/train.py

                    self.last_act = nn.Sigmoid() if n_classes == 1 else nn.ReLU()

                def forward(self, x):
                    x = self.flatten(x); x = x[:, :1536] if x.shape[-1] != 1536 else x; x = self.relu1(self.layernorm1(self.layer1(x)))
                    for block in self.blocks:
                        x = block(x)
                    x = self.last_act(self.last_layer(x))
                    return x
            self.model = Net(input_shape, layer_dim, n_blocks=n_blocks, n_classes=n_classes)
        elif model_type == "rnn":
            class Net(nn.Module):


ERROR: Could not find a version that satisfies the requirement generate_samples (from versions: none)
ERROR: No matching distribution found for generate_samples


In [13]:
import os
import sys
import torch
import types
import yaml
import onnx

# 1. onnx.mapping modülünü belleğe enjekte et (yeni ONNX sürümleri için)
if not hasattr(onnx, "mapping"):
    import onnx.helper
    mapping_mod = types.ModuleType("onnx.mapping")
    mapping_mod.TENSOR_TYPE_TO_NP_TYPE = {}
    mapping_mod.NP_TYPE_TO_TENSOR_TYPE = {}
    for attr in dir(onnx.TensorProto):
        if not attr.startswith("_"):
            val = getattr(onnx.TensorProto, attr)
            if isinstance(val, int):
                try:
                    np_t = onnx.helper.tensor_dtype_to_np_dtype(val)
                    mapping_mod.TENSOR_TYPE_TO_NP_TYPE[val] = np_t
                    mapping_mod.NP_TYPE_TO_TENSOR_TYPE[np_t] = val
                except Exception:
                    pass
    onnx.mapping = mapping_mod
    sys.modules["onnx.mapping"] = mapping_mod
    print("[+] onnx.mapping yaması hazır.")

# 2. YAML ayarlarını oku
config_path = "/content/my_model.yaml"
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

model_name = cfg.get("model_name", "my_model")
output_dir = cfg.get("output_dir", "/content")
os.makedirs(output_dir, exist_ok=True)

# 3. Model mimarisini tanımla
import torch.nn as nn

class FCNBlock(nn.Module):
    def __init__(self, layer_dim):
        super().__init__()
        self.fcn_layer = nn.Linear(layer_dim, layer_dim)
        self.relu = nn.ReLU()
        self.layer_norm = nn.LayerNorm(layer_dim)
    def forward(self, x):
        return self.relu(self.layer_norm(self.fcn_layer(x)))

class Net(nn.Module):
    def __init__(self, input_shape=(16, 96), layer_dim=32, n_blocks=1, n_classes=1):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(input_shape[0]*input_shape[1], layer_dim)
        self.relu1 = nn.ReLU()
        self.layernorm1 = nn.LayerNorm(layer_dim)
        self.blocks = nn.ModuleList([FCNBlock(layer_dim) for _ in range(n_blocks)])
        self.last_layer = nn.Linear(layer_dim, n_classes)
        self.last_act = nn.Sigmoid() if n_classes == 1 else nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        if x.shape[-1] != 1536:
            x = x[:, :1536]
        x = self.relu1(self.layernorm1(self.layer1(x)))
        for block in self.blocks:
            x = block(x)
        x = self.last_act(self.last_layer(x))
        return x

# 4. Kayıtlı en iyi PyTorch model ağırlıklarını ara ve yükle
import glob
possible_weights = glob.glob(f"{output_dir}/**/*.pt", recursive=True) + glob.glob(f"{output_dir}/**/*.pth", recursive=True)

model = Net(input_shape=(16, 96), layer_dim=cfg.get("layer_dim", 32), n_blocks=cfg.get("n_blocks", 1))

if possible_weights:
    latest_weight = max(possible_weights, key=os.path.getmtime)
    print(f"[+] Bulunan ağırlık dosyası yükleniyor: {latest_weight}")
    state = torch.load(latest_weight, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        model.load_state_dict(state["state_dict"])
    elif isinstance(state, dict):
        model.load_state_dict(state)
    else:
        model = state
else:
    print("[!] Kayıtlı .pt dosyası bulunamadı, mevcut mimari dışa aktarılıyor.")

model.eval()

# 5. Doğrudan ONNX formatına dönüştür
onnx_out_path = os.path.join(output_dir, f"{model_name}.onnx")
dummy_input = torch.randn(1, 16, 96)

torch.onnx.export(
    model,
    dummy_input,
    onnx_out_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=13
)

print(f"[✓] Başarılı! ONNX modeli dışa aktarıldı: {onnx_out_path}")

[!] Kayıtlı .pt dosyası bulunamadı, mevcut mimari dışa aktarılıyor.


/tmp/ipykernel_41178/3251822684.py:95: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0922 21:06:41.221000 41178 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
[✓] Başarılı! ONNX modeli dışa aktarıldı: ./my_custom_model/hey_nova.onnx


In [8]:
# Step 3: Train model

!cd /content && PYTHONPATH=/content/openwakeword python /content/openwakeword/openwakeword/train.py --training_config /content/my_model.yaml --train_model

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
Training: 100% 9999/10000 [01:27<00:00, 114.28it/s]
Training: 100% 999/1000.0 [00:09<00:00, 107.16it/s]
Training: 100% 999/1000.0 [00:08<00:00, 113.38it/s]
/content/openwakeword/openwakeword/train.py:429: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(model_to_save.to("cpu"), torch.rand(self.input_shape)[None, ],
W0922 21:03:00.045000 46245 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


In [14]:
import onnxruntime as ort
import numpy as np

# ONNX modelini yükle
session = ort.InferenceSession(onnx_out_path)

# Giriş ve çıkış detaylarını kontrol et
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"Giriş Adı: {input_name}, Şekli: {session.get_inputs()[0].shape}")
print(f"Çıkış Adı: {output_name}, Şekli: {session.get_outputs()[0].shape}")

# 16 karelik (16, 96) dummy ses özniteliği ile test tahmini yap
dummy_feat = np.random.randn(1, 16, 96).astype(np.float32)
pred = session.run([output_name], {input_name: dummy_feat})[0]

print(f"Test Tahmin Skoru: {pred[0][0]:.4f}")
print("✓ Model çıkarım yapmaya hazır!")

Giriş Adı: input, Şekli: ['batch_size', 16, 96]
Çıkış Adı: output, Şekli: ['batch_size', 1]
Test Tahmin Skoru: 0.5762
✓ Model çıkarım yapmaya hazır!


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!